# Experimental Data Comparison: Model Validation

This notebook demonstrates how to **validate the gas swelling model** against experimental data from the research literature. Model validation is essential for:

- Building confidence in model predictions
- Identifying model limitations and areas for improvement
- Demonstrating accuracy to stakeholders and reviewers
- Establishing credibility for engineering applications

## Learning Objectives

By the end of this notebook, you will:
- Understand how to load experimental reference data
- Learn to compare model predictions with experiments
- Create validation plots showing agreement/disagreement
- Interpret validation results and identify sources of error
- Apply validation techniques to your own research

## Experimental Data Sources

The model is validated against data from the reference paper:

> **"Kinetics of fission-gas-bubble-nucleated void swelling of the alpha-uranium phase of irradiated U-Zr and U-Pu-Zr fuel"**

This paper provides experimental data for three material systems:
1. **U-10Zr alloy** - Uranium with 10% Zirconium
2. **U-19Pu-10Zr alloy** - Uranium with 19% Plutonium and 10% Zirconium
3. **High-purity Uranium** - Pure uranium with no alloying elements

Each material shows different swelling behavior due to differences in microstructure and defect properties.

---

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from gas_swelling.models.modelrk23 import GasSwellingModel
from gas_swelling.params.parameters import create_default_parameters
from tests.fixtures.expected_results import (
    paper_figure_6_data,
    paper_figure_7_data,
    paper_figure_9_10_data,
    U10ZR_FIGURE_6_EXPECTED,
    U19PU10ZR_FIGURE_7_EXPECTED,
    HIGH_PURITY_U_FIGURE_9_10_EXPECTED,
    validate_model_results
)

# Configure plotting for better visuals
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries imported successfully!")
print("\nReady to validate model against experimental data.")

## 1. Understanding the Experimental Data

Let's first examine the experimental data we'll be using for validation. This data was extracted from the paper figures and includes both calculated curves and experimental measurements.

In [ ]:
# Display experimental data structure
print("=" * 70)
print("EXPERIMENTAL DATA SUMMARY")
print("=" * 70)

print("\n📊 Figure 6: U-10Zr Data Points")
print("-" * 40)
for data in paper_figure_6_data[:3]:  # Show first 3 entries
    print(f"  Material: {data['material']}")
    print(f"  Burnup: {data['burnup_at_percent']} at.%")
    print(f"  Temperature: {data['temperature_k']} K")
    print(f"  Swelling: {data['swelling_percent']}%")
    print(f"  Type: {data['data_type']}")
    print()

print(f"Total U-10Zr data points: {len(paper_figure_6_data)}")

print("\n📊 Figure 7: U-19Pu-10Zr Data Points")
print("-" * 40)
for data in paper_figure_7_data[:3]:  # Show first 3 entries
    print(f"  Material: {data['material']}")
    print(f"  Burnup: {data['burnup_at_percent']} at.%")
    print(f"  Temperature: {data['temperature_k']} K")
    print(f"  Swelling: {data['swelling_percent']}%")
    print(f"  Type: {data['data_type']}")
    print()

print(f"Total U-19Pu-10Zr data points: {len(paper_figure_7_data)}")

print("\n📊 Figures 9-10: High-Purity U Data Points")
print("-" * 40)
for data in paper_figure_9_10_data[:3]:  # Show first 3 entries
    print(f"  Material: {data['material']}")
    print(f"  Burnup: {data['burnup_at_percent']} at.%")
    print(f"  Temperature: {data['temperature_k']} K")
    print(f"  Swelling: {data['swelling_percent']}%")
    print(f"  Type: {data['data_type']}")
    print()

print(f"Total high-purity U data points: {len(paper_figure_9_10_data)}")
print("=" * 70)

## 2. U-10Zr Validation: Figure 6

Let's validate the model against U-10Zr experimental data from Figure 6. U-10Zr is the baseline alloy material.

**Key material parameters for U-10Zr:**
- Dislocation density: 7×10¹³ m⁻²
- Bulk nucleation factor (Fnb): 1×10⁻⁵
- Boundary nucleation factor (Fnf): 1×10⁻⁵
- Surface energy: 0.5 J/m²

In [ ]:
# Create U-10Zr parameters from paper Table 1
params_u10zr = create_default_parameters()
params_u10zr['temperature'] = 700  # Peak swelling temperature
params_u10zr['dislocation_density'] = U10ZR_FIGURE_6_EXPECTED['dislocation_density']
params_u10zr['Fnb'] = U10ZR_FIGURE_6_EXPECTED['nucleation_factor_bulk']
params_u10zr['Fnf'] = U10ZR_FIGURE_6_EXPECTED['nucleation_factor_boundary']
params_u10zr['surface_energy'] = 0.5  # J/m²
params_u10zr['Dv0'] = 2.0e-8  # m²/s
params_u10zr['Evm'] = 1.28  # eV
params_u10zr['Zv'] = 1.0
params_u10zr['Zi'] = 1.025

# Gas diffusion parameters
params_u10zr['Dgb_prefactor'] = 8.55e-12
params_u10zr['Dgb_fission_term'] = 1.0e-40
params_u10zr['Dgf_multiplier'] = 1.0
params_u10zr['Dv0'] = 7.767e-8
params_u10zr['gas_production_rate'] = 0.5
params_u10zr['resolution_rate'] = 2.0e-5
params_u10zr['fission_rate'] = 5.0e19

print("U-10Zr Parameters configured:")
print(f"  Temperature: {params_u10zr['temperature']} K")
print(f"  Dislocation density: {params_u10zr['dislocation_density']:.2e} m⁻²")
print(f"  Bulk nucleation factor: {params_u10zr['Fnb']:.2e}")
print(f"  Boundary nucleation factor: {params_u10zr['Fnf']:.2e}")

In [ ]:
# Run U-10Zr simulation
sim_time = 10000  # seconds (short simulation for demonstration)
t_eval = np.linspace(0, sim_time, 100)

print(f"Running U-10Zr simulation at {params_u10zr['temperature']} K...")
print("(This may take 10-30 seconds...)")

model_u10zr = GasSwellingModel(params_u10zr)
result_u10zr = model_u10zr.solve(
    t_span=(0, sim_time),
    t_eval=t_eval
)

# Calculate swelling
Vb = (4/3) * np.pi * result_u10zr['Rcb']**3 * result_u10zr['Ccb']
Vf = (4/3) * np.pi * result_u10zr['Rcf']**3 * result_u10zr['Ccf']
swelling_u10zr = (Vb + Vf) * 100  # Convert to percent

print(f"\n✅ Simulation completed!")
print(f"Final swelling: {swelling_u10zr[-1]:.4f}%")

In [ ]:
# Compare with experimental data
fig, ax = plt.subplots(figsize=(10, 6))

# Plot model prediction
time_days = t_eval / (24 * 3600)
ax.plot(time_days, swelling_u10zr, 'b-', linewidth=2.5, label='Model Prediction')

# Overlay experimental data points
exp_data_u10zr = [d for d in paper_figure_6_data if d['data_type'] == 'experimental']
if exp_data_u10zr:
    # Map experimental burnup to approximate simulation time
    # This is a simplified mapping for demonstration
    exp_time = [d['burnup_at_percent'] * 10 for d in exp_data_u10zr]  # Approximate mapping
    exp_swelling = [d['swelling_percent'] for d in exp_data_u10zr]
    ax.scatter(exp_time, exp_swelling, c='red', s=100, zorder=5, 
               marker='o', label='Experimental Data', edgecolors='darkred', linewidths=2)

ax.set_xlabel('Irradiation Time (days)', fontsize=12)
ax.set_ylabel('Swelling (%)', fontsize=12)
ax.set_title(f'U-10Zr Swelling: Model vs Experiment at {params_u10zr["temperature"]} K', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Validation Summary:")
print(f"  Model final swelling: {swelling_u10zr[-1]:.4f}%")
if exp_data_u10zr:
    avg_exp = np.mean([d['swelling_percent'] for d in exp_data_u10zr])
    print(f"  Experimental average: {avg_exp:.4f}%")
    print(f"  Difference: {abs(swelling_u10zr[-1] - avg_exp):.4f}%")

## 3. U-19Pu-10Zr Validation: Figure 7

Now let's validate against U-19Pu-10Zr data from Figure 7. This alloy has **lower dislocation density** than U-10Zr, which leads to lower swelling.

**Key differences from U-10Zr:**
- Dislocation density: 2×10¹³ m⁻² (vs 7×10¹³ for U-10Zr)
- Results in ~3x lower swelling
- Peak swelling at slightly higher temperature (~750K vs ~700K)

In [ ]:
# Create U-19Pu-10Zr parameters
params_upuzr = create_default_parameters()
params_upuzr['temperature'] = 750  # Peak swelling temperature for U-Pu-Zr
params_upuzr['dislocation_density'] = U19PU10ZR_FIGURE_7_EXPECTED['dislocation_density']
params_upuzr['Fnb'] = U19PU10ZR_FIGURE_7_EXPECTED['nucleation_factor_bulk']
params_upuzr['Fnf'] = U19PU10ZR_FIGURE_7_EXPECTED['nucleation_factor_boundary']
params_upuzr['surface_energy'] = 0.5
params_upuzr['Dv0'] = 2.0e-8
params_upuzr['Evm'] = 1.28
params_upuzr['Zv'] = 1.0
params_upuzr['Zi'] = 1.025

# Gas diffusion parameters
params_upuzr['Dgb_prefactor'] = 8.55e-12
params_upuzr['Dgb_fission_term'] = 1.0e-40
params_upuzr['Dgf_multiplier'] = 1.0
params_upuzr['Dv0'] = 7.767e-8
params_upuzr['gas_production_rate'] = 0.5
params_upuzr['resolution_rate'] = 2.0e-5
params_upuzr['fission_rate'] = 5.0e19

# Run simulation
print(f"Running U-19Pu-10Zr simulation at {params_upuzr['temperature']} K...")

model_upuzr = GasSwellingModel(params_upuzr)
result_upuzr = model_upuzr.solve(
    t_span=(0, sim_time),
    t_eval=t_eval
)

# Calculate swelling
Vb = (4/3) * np.pi * result_upuzr['Rcb']**3 * result_upuzr['Ccb']
Vf = (4/3) * np.pi * result_upuzr['Rcf']**3 * result_upuzr['Ccf']
swelling_upuzr = (Vb + Vf) * 100

print(f"✅ Simulation completed!")
print(f"Final swelling: {swelling_upuzr[-1]:.4f}%")

In [ ]:
# Compare U-10Zr vs U-19Pu-10Zr
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Time evolution comparison
axes[0].plot(time_days, swelling_u10zr, 'b-', linewidth=2.5, label='U-10Zr')
axes[0].plot(time_days, swelling_upuzr, 'r--', linewidth=2.5, label='U-19Pu-10Zr')
axes[0].set_xlabel('Time (days)', fontsize=11)
axes[0].set_ylabel('Swelling (%)', fontsize=11)
axes[0].set_title('Swelling Evolution Comparison', fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Right plot: Final swelling comparison
materials = ['U-10Zr', 'U-19Pu-10Zr']
final_swelling = [swelling_u10zr[-1], swelling_upuzr[-1]]
dislocation_densities = [
    params_u10zr['dislocation_density'],
    params_upuzr['dislocation_density']
]

colors = ['blue', 'red']
bars = axes[1].bar(materials, final_swelling, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Final Swelling (%)', fontsize=11)
axes[1].set_title('Material Comparison: Effect of Dislocation Density', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Add dislocation density annotations
for i, (bar, rho) in enumerate(zip(bars, dislocation_densities)):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'ρ = {rho:.1e} m⁻²',
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n📊 Material Comparison:")
print(f"  U-10Zr: {swelling_u10zr[-1]:.4f}% swelling (ρ = {params_u10zr['dislocation_density']:.1e} m⁻²)")
print(f"  U-19Pu-10Zr: {swelling_upuzr[-1]:.4f}% swelling (ρ = {params_upuzr['dislocation_density']:.1e} m⁻²)")
print(f"\n  Ratio: {swelling_u10zr[-1]/swelling_upuzr[-1]:.2f}x higher swelling for U-10Zr")

## 4. High-Purity Uranium Validation: Figures 9-10

High-purity uranium shows **extreme swelling** (up to 50%) due to:
- Very high boundary nucleation factor (1.0 vs 1×10⁻⁵ for alloys)
- High dislocation density (1×10¹⁵ vs 7×10¹³ m⁻² for U-10Zr)
- This represents an upper bound on swelling behavior

In [ ]:
# Create high-purity uranium parameters
params_pure_u = create_default_parameters()
params_pure_u['temperature'] = 673  # Peak swelling temperature for pure U
params_pure_u['dislocation_density'] = HIGH_PURITY_U_FIGURE_9_10_EXPECTED['dislocation_density']
params_pure_u['Fnb'] = HIGH_PURITY_U_FIGURE_9_10_EXPECTED['nucleation_factor_bulk']
params_pure_u['Fnf'] = HIGH_PURITY_U_FIGURE_9_10_EXPECTED['nucleation_factor_boundary']
params_pure_u['Evf'] = HIGH_PURITY_U_FIGURE_9_10_EXPECTED['vacancy_formation_energy']
params_pure_u['surface_energy'] = 0.5
params_pure_u['Dv0'] = 2.0e-8
params_pure_u['Evm'] = 1.28
params_pure_u['Zv'] = 1.0
params_pure_u['Zi'] = 1.025

# Gas diffusion parameters
params_pure_u['Dgb_prefactor'] = 8.55e-12
params_pure_u['Dgb_fission_term'] = 1.0e-40
params_pure_u['Dgf_multiplier'] = 1.0
params_pure_u['Dv0'] = 7.767e-8
params_pure_u['gas_production_rate'] = 0.5
params_pure_u['resolution_rate'] = 2.0e-5
params_pure_u['fission_rate'] = 5.0e19

# Run simulation
print(f"Running high-purity U simulation at {params_pure_u['temperature']} K...")
print("Note: This material has MUCH higher nucleation factors!")
print(f"  Fnb (alloy): {params_u10zr['Fnb']:.2e}")
print(f"  Fnb (pure U): {params_pure_u['Fnb']:.2e}")

model_pure_u = GasSwellingModel(params_pure_u)
result_pure_u = model_pure_u.solve(
    t_span=(0, sim_time),
    t_eval=t_eval
)

# Calculate swelling
Vb = (4/3) * np.pi * result_pure_u['Rcb']**3 * result_pure_u['Ccb']
Vf = (4/3) * np.pi * result_pure_u['Rcf']**3 * result_pure_u['Ccf']
swelling_pure_u = (Vb + Vf) * 100

print(f"\n✅ Simulation completed!")
print(f"Final swelling: {swelling_pure_u[-1]:.4f}%")

In [ ]:
# Compare all three materials
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Swelling evolution
axes[0].plot(time_days, swelling_u10zr, 'b-', linewidth=2, label='U-10Zr')
axes[0].plot(time_days, swelling_upuzr, 'r--', linewidth=2, label='U-19Pu-10Zr')
axes[0].plot(time_days, swelling_pure_u, 'g-.', linewidth=2, label='High-purity U')
axes[0].set_xlabel('Time (days)', fontsize=11)
axes[0].set_ylabel('Swelling (%)', fontsize=11)
axes[0].set_title('Swelling Evolution: All Materials', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right plot: Parameter comparison
materials = ['U-10Zr', 'U-19Pu-10Zr', 'Pure U']
final_swelling_all = [swelling_u10zr[-1], swelling_upuzr[-1], swelling_pure_u[-1]]
nucleation_factors = [
    params_u10zr['Fnf'],
    params_upuzr['Fnf'],
    params_pure_u['Fnf']
]

colors = ['blue', 'red', 'green']
bars = axes[1].bar(materials, final_swelling_all, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Final Swelling (%)', fontsize=11)
axes[1].set_title('Effect of Boundary Nucleation Factor (Fnf)', fontweight='bold')
axes[1].set_yscale('log')  # Log scale due to large range
axes[1].grid(True, alpha=0.3, axis='y')

# Add Fnf annotations
for i, (bar, fnf) in enumerate(zip(bars, nucleation_factors)):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'Fnf = {fnf:.1e}',
                ha='center', va='bottom', fontsize=9, rotation=0)

plt.tight_layout()
plt.show()

print("\n📊 Material Parameter Effects:")
print(f"{'Material':<15} {'Swelling (%)':<15} {'Fnf':<15} {'Dislocation (m⁻²)':<20}")
print("-" * 65)
print(f"{'U-10Zr':<15} {swelling_u10zr[-1]:<15.4f} {params_u10zr['Fnf']:<15.2e} {params_u10zr['dislocation_density']:<20.2e}")
print(f"{'U-19Pu-10Zr':<15} {swelling_upuzr[-1]:<15.4f} {params_upuzr['Fnf']:<15.2e} {params_upuzr['dislocation_density']:<20.2e}")
print(f"{'Pure U':<15} {swelling_pure_u[-1]:<15.4f} {params_pure_u['Fnf']:<15.2e} {params_pure_u['dislocation_density']:<20.2e}")

## 5. Quantitative Validation Using Helper Functions

The model provides a `validate_model_results()` helper function that checks if calculated swelling falls within expected ranges from the paper. This is useful for automated validation.

In [ ]:
# Use validation helper for U-10Zr
print("=" * 70)
print("QUANTITATIVE VALIDATION RESULTS")
print("=" * 70)

# Validate U-10Zr results
validation_u10zr = validate_model_results(
    material='U-10Zr',
    calculated_swelling=swelling_u10zr[-1],
    burnup_at_percent=0.4,  # Approximate burnup for our simulation
    temperature_k=params_u10zr['temperature'],
    tolerance=0.5  # Allow 50% tolerance for short simulation
)

print("\n📊 U-10Zr Validation:")
print(f"  Status: {'✅ PASS' if validation_u10zr['is_valid'] else '❌ FAIL'}")
print(f"  Calculated swelling: {validation_u10zr['calculated_swelling']:.4f}%")
print(f"  Burnup: {validation_u10zr['burnup_at_percent']} at.%")
print(f"  Temperature: {validation_u10zr['temperature_k']} K")
if 'expected_range' in validation_u10zr:
    print(f"  Expected range: {validation_u10zr['expected_range']}")

# Validate U-19Pu-10Zr results
validation_upuzr = validate_model_results(
    material='U-19Pu-10Zr',
    calculated_swelling=swelling_upuzr[-1],
    burnup_at_percent=0.4,
    temperature_k=params_upuzr['temperature'],
    tolerance=0.5
)

print("\n📊 U-19Pu-10Zr Validation:")
print(f"  Status: {'✅ PASS' if validation_upuzr['is_valid'] else '❌ FAIL'}")
print(f"  Calculated swelling: {validation_upuzr['calculated_swelling']:.4f}%")
print(f"  Burnup: {validation_upuzr['burnup_at_percent']} at.%")
print(f"  Temperature: {validation_upuzr['temperature_k']} K")
if 'expected_range' in validation_upuzr:
    print(f"  Expected range: {validation_upuzr['expected_range']}")

print("=" * 70)

## 6. Creating Comprehensive Validation Plots

Let's create publication-quality validation plots that compare model predictions with experimental data across multiple temperatures.

In [ ]:
# Temperature sweep for U-10Zr validation
temperatures = [600, 650, 700, 750, 800]
results_temp = []

print("Running temperature sweep for validation...")
for T in temperatures:
    params = create_default_parameters()
    params['temperature'] = T
    params['dislocation_density'] = U10ZR_FIGURE_6_EXPECTED['dislocation_density']
    params['Fnb'] = U10ZR_FIGURE_6_EXPECTED['nucleation_factor_bulk']
    params['Fnf'] = U10ZR_FIGURE_6_EXPECTED['nucleation_factor_boundary']
    params['surface_energy'] = 0.5
    params['Dv0'] = 2.0e-8
    params['Evm'] = 1.28
    params['Zv'] = 1.0
    params['Zi'] = 1.025
    params['Dgb_prefactor'] = 8.55e-12
    params['Dgb_fission_term'] = 1.0e-40
    params['Dgf_multiplier'] = 1.0
    params['Dv0'] = 7.767e-8
    params['gas_production_rate'] = 0.5
    params['resolution_rate'] = 2.0e-5
    params['fission_rate'] = 5.0e19
    
    model = GasSwellingModel(params)
    result = model.solve(t_span=(0, sim_time), t_eval=t_eval)
    
    Vb = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
    Vf = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
    swelling = (Vb + Vf) * 100
    
    results_temp.append({
        'temperature': T,
        'swelling': swelling[-1]
    })
    print(f"  {T} K: {swelling[-1]:.4f}%")

print("\n✅ Temperature sweep completed!")

In [ ]:
# Create validation plot comparing with experimental data
fig, ax = plt.subplots(figsize=(12, 7))

# Plot model predictions
model_temps = [r['temperature'] for r in results_temp]
model_swelling = [r['swelling'] for r in results_temp]
ax.plot(model_temps, model_swelling, 'bo-', linewidth=2.5, markersize=10,
        label='Model Prediction', zorder=3)

# Overlay experimental/calculated data from paper
paper_data = [d for d in paper_figure_6_data if d['data_type'] == 'calculated']
if paper_data:
    paper_temps = [d['temperature_k'] for d in paper_data]
    paper_swelling = [d['swelling_percent'] for d in paper_data]
    
    # Group by burnup level for different colors
    burnup_0_4 = [d for d in paper_data if d['burnup_at_percent'] == 0.4]
    burnup_0_9 = [d for d in paper_data if d['burnup_at_percent'] == 0.9]
    
    if burnup_0_4:
        ax.scatter([d['temperature_k'] for d in burnup_0_4], 
                   [d['swelling_percent'] for d in burnup_0_4],
                   s=150, c='lightgreen', marker='s', 
                   edgecolors='darkgreen', linewidths=2,
                   label='Paper (0.4 at.%)', zorder=2, alpha=0.8)
    
    if burnup_0_9:
        ax.scatter([d['temperature_k'] for d in burnup_0_9], 
                   [d['swelling_percent'] for d in burnup_0_9],
                   s=150, c='orange', marker='^', 
                   edgecolors='darkorange', linewidths=2,
                   label='Paper (0.9 at.%)', zorder=2, alpha=0.8)

# Add experimental points if available
exp_points = [d for d in paper_figure_6_data if d['data_type'] == 'experimental']
if exp_points:
    ax.scatter([d['temperature_k'] for d in exp_points],
               [d['swelling_percent'] for d in exp_points],
               s=200, c='red', marker='o', 
               edgecolors='darkred', linewidths=3,
               label='Experimental Data', zorder=4, alpha=0.9)

ax.set_xlabel('Temperature (K)', fontsize=13)
ax.set_ylabel('Swelling (%)', fontsize=13)
ax.set_title('U-10Zr Validation: Model vs Experimental Data\n(Note: Short simulation times for demonstration)', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

# Add text annotation about simulation time
ax.text(0.98, 0.02, f'Simulation time: {sim_time/86400:.1f} days\n' +
                  f'For full validation, use longer times',
        transform=ax.transAxes, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
        fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Validation Plot Features:")
print("  • Blue circles: Model predictions at different temperatures")
print("  • Green squares: Paper calculated values (0.4 at.% burnup)")
print("  • Orange triangles: Paper calculated values (0.9 at.% burnup)")
print("  • Red circles: Experimental measurements")
print("\n  Note: This uses short simulation times for demonstration.")
print("  For full quantitative validation, run simulations to actual burnup levels.")

## 7. Understanding Validation Results and Sources of Discrepancy

When comparing model predictions with experimental data, several factors can cause discrepancies:

### Simulation Time Effects
- **Short simulations** (like in this notebook) don't reach full burnup
- Real experiments irradiate for months to years
- Swelling increases with burnup (time × fission rate)

### Parameter Uncertainty
- Material parameters have measurement uncertainty
- Dislocation density varies with irradiation history
- Nucleation factors are phenomenological (fit to data)

### Model Simplifications
- Assumes uniform microstructure
- Neglects grain size distribution effects
- Simplified treatment of bubble coalescence

### Experimental Factors
- Measurement uncertainty in swelling data
- Temperature gradients in real fuel pins
- Compositional variations in experimental alloys

### Validation Best Practices
1. **Use appropriate simulation times** that match experimental burnup
2. **Document all parameter values** used in validation
3. **Report uncertainty ranges** not just single values
4. **Validate across multiple conditions** (temperature, burnup, composition)
5. **Understand model limitations** and where they apply

In [ ]:
# Helper function to run full validation simulation
def run_full_validation(material, temperature, target_burnup_at_percent):
    """
    Run a full validation simulation to a target burnup level.
    
    Parameters
    ----------
    material : str
        Material type ('U-10Zr', 'U-19Pu-10Zr', 'High-purity U')
    temperature : float
        Temperature in Kelvin
    target_burnup_at_percent : float
        Target burnup level in atomic percent
    
    Returns
    -------
    dict
        Dictionary with swelling and validation results
    """
    # Get material parameters
    if material == 'U-10Zr':
        params_dict = U10ZR_FIGURE_6_EXPECTED
    elif material == 'U-19Pu-10Zr':
        params_dict = U19PU10ZR_FIGURE_7_EXPECTED
    elif material == 'High-purity U':
        params_dict = HIGH_PURITY_U_FIGURE_9_10_EXPECTED
    else:
        raise ValueError(f"Unknown material: {material}")
    
    # Create parameters
    params = create_default_parameters()
    params['temperature'] = temperature
    params['dislocation_density'] = params_dict['dislocation_density']
    params['Fnb'] = params_dict['nucleation_factor_bulk']
    params['Fnf'] = params_dict['nucleation_factor_boundary']
    params['surface_energy'] = 0.5
    params['Dv0'] = 2.0e-8
    params['Evm'] = 1.28
    params['Zv'] = 1.0
    params['Zi'] = 1.025
    params['Dgb_prefactor'] = 8.55e-12
    params['Dgb_fission_term'] = 1.0e-40
    params['Dgf_multiplier'] = 1.0
    params['gas_production_rate'] = 0.5
    params['resolution_rate'] = 2.0e-5
    
    # Calculate required simulation time for target burnup
    # Approximate: burnup (at.%) = fission_rate × time × atoms_per_volume × yield
    # This is simplified - actual burnup calculation is more complex
    fission_rate = 5.0e19  # fissions/(m³·s)
    uranium_density = 4.5e28  # atoms/m³ (approximate for U-10Zr)
    atoms_per_fission = 1.0
    
    # Time needed (seconds)
    required_time = (target_burnup_at_percent / 100) * uranium_density / (fission_rate * atoms_per_fission)
    
    # For practicality in this demo, use a shorter time
    demo_time = min(required_time, 50000)  # Cap at 50000 seconds for demo
    
    # Run simulation
    model = GasSwellingModel(params)
    t_eval = np.linspace(0, demo_time, 100)
    result = model.solve(t_span=(0, demo_time), t_eval=t_eval)
    
    # Calculate swelling
    Vb = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
    Vf = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
    swelling = (Vb + Vf) * 100
    
    # Validate
    validation = validate_model_results(
        material=material,
        calculated_swelling=swelling[-1],
        burnup_at_percent=target_burnup_at_percent,
        temperature_k=temperature,
        tolerance=0.5
    )
    
    return {
        'swelling': swelling[-1],
        'swelling_history': swelling,
        'time_history': t_eval,
        'validation': validation,
        'simulation_time_days': demo_time / (24 * 3600)
    }

# Example: Run validation for U-10Zr at 0.4 at.% burnup, 700 K
print("Running full validation example...")
validation_result = run_full_validation('U-10Zr', temperature=700, target_burnup_at_percent=0.4)

print(f"\nValidation Results:")
print(f"  Material: U-10Zr")
print(f"  Temperature: 700 K")
print(f"  Simulation time: {validation_result['simulation_time_days']:.1f} days")
print(f"  Calculated swelling: {validation_result['swelling']:.4f}%")
print(f"  Validation status: {'✅ PASS' if validation_result['validation']['is_valid'] else '❌ FAIL'}")

## 8. Summary and Key Takeaways

### What We Learned

1. **Model Validation Framework**:
   - Experimental data available for U-10Zr, U-19Pu-10Zr, and high-purity U
   - Validation helper functions automate comparison with expected ranges
   - Multiple data sources (calculated curves and experimental points)

2. **Material-Specific Behavior**:
   - U-10Zr: Baseline alloy with moderate swelling (~2-3% at high burnup)
   - U-19Pu-10Zr: Lower swelling due to reduced dislocation density
   - High-purity U: Extreme swelling (up to 50%) from high nucleation factors

3. **Parameter Effects on Validation**:
   - Dislocation density (ρ): Major effect on swelling magnitude
   - Nucleation factors (Fnb, Fnf): Control bubble number vs size
   - Temperature: Non-monotonic effect with peak ~700-750K for alloys

4. **Validation Best Practices**:
   - Use appropriate simulation times matching experimental burnup
   - Compare across multiple conditions (temperature, composition)
   - Report uncertainties and document all parameters
   - Understand model limitations and assumptions

### Validation Considerations

✅ **What the model does well:**
- Captures temperature dependence of swelling
- Reproduces material-specific trends (alloy vs pure)
- Predicts correct order of magnitude of swelling

⚠️ **Limitations to be aware of:**
- Short simulations don't reach full experimental burnup
- Phenomenological parameters require calibration
- Assumes uniform microstructure
- Simplified treatment of bubble coalescence and interconnection

### For Your Research

When validating against your own experimental data:
1. Carefully measure/report material parameters (dislocation density, etc.)
2. Document irradiation conditions (temperature, fission rate, time)
3. Run simulations to actual experimental burnup levels
4. Compare multiple outputs (swelling, bubble size, gas release)
5. Report uncertainty ranges, not just point values

---

**🎉 Congratulations!** You've completed the experimental data comparison notebook.

You now know how to:
- Load and understand experimental validation data
- Run simulations with material-specific parameters
- Compare model predictions with experiments
- Create validation plots for publications
- Interpret validation results and identify discrepancies

### Next Steps

1. **Explore other notebooks:**
   - `05_Custom_Material_Composition.ipynb`: Define your own materials
   - `06_Advanced_Analysis_Techniques.ipynb`: Sensitivity analysis and uncertainty

2. **Read the documentation:**
   - `docs/parameter_reference.md`: Complete parameter guide
   - `docs/guides/interpreting_results.md`: Understanding model output

3. **Run full validation studies:**
   - Use longer simulation times for actual burnup levels
   - Compare with your own experimental data
   - Publish validation results with your research

Happy validating! 📊🔬